# Notebook 6: Gaussian Mixture Models (GMM) & the EM Algorithm

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 75 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Understand **soft cluster assignments** — probabilistic membership in multiple clusters simultaneously
2. Implement the **EM algorithm** for GMMs from scratch (E-step and M-step)
3. Explain what happens when EM converges and what the log-likelihood curve looks like
4. Compare GMM vs. K-means for elliptical and overlapping clusters

---

**Week 11 Theme:** Customer segmentation / retail data


## The Intuition: Blended Customer Segments

K-means tells you: *"Customer A belongs to Segment 2."* Full stop.

But real customers are messy. Customer A might shop like a budget buyer most months, occasionally splurge like a luxury buyer, and sometimes look like a deal-seeker. Forcing a single hard label loses that nuance.

**Gaussian Mixture Models** take a different approach:

> "Customer A is **70% luxury buyer**, **25% deal-seeker**, and **5% occasional shopper**."

Each segment is modelled as a **Gaussian blob** (a cloud of points with a centre and a shape). The model finds the best-fitting blend of Gaussians to explain the observed data — and tells you how much each customer belongs to each blob.

This is exactly what a **mixture model** is: a weighted sum of probability distributions, where the weights tell you how likely a randomly chosen customer came from each component.


## Why Does This Matter for ML?

GMMs are important for four distinct reasons:

| Reason | Practical impact |
|--------|------------------|
| **Soft assignments** | Model uncertainty — a customer near a segment boundary gets near-equal probabilities for both |
| **Elliptical clusters** | Full covariance matrices let clusters be elongated, rotated, or tilted — not just round |
| **Principled probabilistic framework** | You get likelihoods, BIC/AIC for model selection, and the ability to generate new data |
| **Generative model** | Once fitted, you can *sample* new customers from each segment |

In retail segmentation, soft assignments are particularly valuable: they let you target a customer with the product mix proportional to their segment probabilities, rather than picking one segment and ignoring the others.


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import seaborn as sns

from scipy.stats import multivariate_normal
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs

np.random.seed(42)                    # reproducibility
sns.set_theme(style='whitegrid')      # consistent plot style

print("Imports complete.")


## The GMM Model — Formal Description

### What a mixture model says about your data

A GMM assumes your data was generated by the following two-step process:

1. **Pick a component** k with probability π_k (the **mixing weight**)
2. **Sample a point** from the k-th Gaussian: x ~ N(μ_k, Σ_k)

You observe only x — not which component it came from. Your job is to infer the parameters (π_k, μ_k, Σ_k) for each component from the data alone.

### The joint density

Plain English: *"The probability of observing a point x is the weighted average of how probable x is under each individual Gaussian."*

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

where:
- $K$ = number of components (you choose this, like k in K-means)
- $\pi_k \geq 0$, $\sum_k \pi_k = 1$ — mixing weights (must sum to 1)
- $\boldsymbol{\mu}_k$ — mean of component k
- $\boldsymbol{\Sigma}_k$ — covariance matrix of component k (controls shape and orientation)

### Parameters to learn

For K components in d dimensions: K means (d-dimensional), K covariance matrices (d×d), and K−1 free mixing weights. We estimate these by **maximising the log-likelihood** of the observed data.


In [ ]:
np.random.seed(42)

# ── True GMM parameters ───────────────────────────────────────────────────────
TRUE_MEANS = np.array([
    [0.0,  0.0],    # component 0: centre of space
    [5.0,  1.0],    # component 1: right of space
    [2.5,  5.0],    # component 2: top of space
])

TRUE_COVS = [
    np.array([[1.0, 0.5], [0.5, 0.6]]),    # tilted ellipse
    np.array([[0.8, -0.3], [-0.3, 1.2]]),  # tilted ellipse (opposite direction)
    np.array([[1.5, 0.0], [0.0, 0.4]]),    # horizontal ellipse
]

TRUE_WEIGHTS = np.array([0.35, 0.40, 0.25])  # must sum to 1
N_TOTAL = 600

# ── Sample data from the true GMM ────────────────────────────────────────────
component_samples = np.random.multinomial(N_TOTAL, TRUE_WEIGHTS)  # how many from each
X_parts = [
    np.random.multivariate_normal(TRUE_MEANS[k], TRUE_COVS[k], component_samples[k])
    for k in range(3)
]
true_component_ids = np.concatenate([
    np.full(component_samples[k], k) for k in range(3)
])
X_gmm = np.vstack(X_parts)   # combined dataset, labels hidden

# ── Shuffle so the true labels are not obvious by row order ──────────────────
perm = np.random.permutation(N_TOTAL)
X_gmm = X_gmm[perm]
true_component_ids = true_component_ids[perm]

# ── Helper: draw covariance ellipse ──────────────────────────────────────────
def draw_cov_ellipse(ax, mean, cov, n_std=1.0, color='black', linewidth=2, linestyle='-'):
    """Draw an n_std-sigma covariance ellipse on ax."""
    vals, vecs = np.linalg.eigh(cov)   # eigenvalues and eigenvectors
    order = vals.argsort()[::-1]       # sort descending
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))  # rotation angle in degrees
    width, height = 2 * n_std * np.sqrt(vals)           # axes lengths
    ellipse = Ellipse(
        xy=mean, width=width, height=height, angle=angle,
        edgecolor=color, fc='none', linewidth=linewidth, linestyle=linestyle
    )
    ax.add_patch(ellipse)

# ── Plot 1: raw data (no labels) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1], s=15, alpha=0.4, color='steelblue')
axes[0].set_title('Raw Data — No Labels (What the Algorithm Sees)', fontsize=12)
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')

# Plot 2: true labels + Gaussian contours
pal = ['#4C72B0', '#DD8452', '#55A868']
for k in range(3):
    mask = true_component_ids == k
    axes[1].scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                    c=pal[k], s=15, alpha=0.5, label=f'Component {k}')
    draw_cov_ellipse(axes[1], TRUE_MEANS[k], TRUE_COVS[k], n_std=1, color=pal[k], linewidth=2)
    draw_cov_ellipse(axes[1], TRUE_MEANS[k], TRUE_COVS[k], n_std=2, color=pal[k], linewidth=1, linestyle='--')

axes[1].set_title('True Labels + 1σ/2σ Gaussian Contours', fontsize=12)
axes[1].set_xlabel('Feature 1'); axes[1].set_ylabel('Feature 2')
axes[1].legend()

fig.suptitle('GMM Synthetic Data (3 components)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Dataset shape: {X_gmm.shape}")
print(f"True component sizes: {[int((true_component_ids==k).sum()) for k in range(3)]}")


## Hard vs. Soft Assignments

Here is the key conceptual difference between K-means and GMM:

### K-means: Hard Assignment

Each point gets a single integer label. There is no uncertainty — point i either belongs to cluster 0 or cluster 1 or cluster 2, never both.

```
Point i → cluster label: 2
```

### GMM: Soft Assignment (Responsibilities)

Each point gets a probability vector — how responsible is each component for generating this point?

```
Point i → responsibilities: [0.05, 0.70, 0.25]
```

This vector is called **r_ik** — the responsibility of component k for point i.

Formally, it is a posterior probability (Bayes' theorem):

$$r_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

Plain English: *"The responsibility of component k for point x_i is proportional to how likely k is (π_k) times how well k explains x_i (the Gaussian density)."*

Note: r_ik values are always ≥ 0 and sum to 1 across k for every point i.


In [ ]:
np.random.seed(42)

# ── Illustrate soft assignments on 5 selected points ─────────────────────────
# Use the true parameters to compute responsibilities

def compute_responsibilities(X, means, covs, weights):
    """E-step: compute responsibility matrix R of shape (n, K)."""
    K = len(means)
    n = len(X)
    R = np.zeros((n, K))
    for k in range(K):
        # Weight each Gaussian density by the mixing weight
        R[:, k] = weights[k] * multivariate_normal.pdf(X, means[k], covs[k])
    # Normalise each row to sum to 1 (Bayes' rule denominator)
    row_sums = R.sum(axis=1, keepdims=True)
    R /= row_sums + 1e-300   # add tiny constant to avoid division by zero
    return R

R_true = compute_responsibilities(X_gmm, TRUE_MEANS, TRUE_COVS, TRUE_WEIGHTS)

# Pick 5 interesting points: one clearly inside each cluster, one on a boundary
# Find point closest to each component mean, plus one midway between 0 and 1
selected_indices = []
for k in range(3):
    dists = np.linalg.norm(X_gmm - TRUE_MEANS[k], axis=1)
    selected_indices.append(np.argmin(dists))   # closest to each mean

# One boundary point: highest entropy (most uncertain)
entropy = -np.sum(R_true * np.log(R_true + 1e-10), axis=1)
selected_indices.append(np.argmax(entropy))     # most uncertain point
selected_indices.append(np.argsort(entropy)[-2])  # second most uncertain

# ── Bar charts showing responsibilities for each selected point ───────────────
fig, axes = plt.subplots(1, 5, figsize=(15, 4))

for ax_idx, pt_idx in enumerate(selected_indices):
    responsibilities = R_true[pt_idx]
    bars = axes[ax_idx].bar(
        ['Comp 0', 'Comp 1', 'Comp 2'],
        responsibilities,
        color=pal, edgecolor='white'
    )
    axes[ax_idx].set_ylim(0, 1.05)
    axes[ax_idx].set_title(
        f'Point {pt_idx}\n({X_gmm[pt_idx, 0]:.1f}, {X_gmm[pt_idx, 1]:.1f})',
        fontsize=9
    )
    axes[ax_idx].set_ylabel('Responsibility r_ik' if ax_idx == 0 else '')
    # Label the dominant component
    dominant = np.argmax(responsibilities)
    axes[ax_idx].set_xlabel(f'Dominant: Comp {dominant}', fontsize=9)
    for bar, val in zip(bars, responsibilities):
        axes[ax_idx].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                          f'{val:.2f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Soft Assignments: Responsibility r_ik for 5 Selected Points', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Points near a mean → responsibility ≈ 1.0 for one component")
print("Points on boundaries → responsibilities spread across components")


## The EM Algorithm for GMMs

We cannot solve for the GMM parameters in closed form because we do not know which component generated each point (the **latent variable**). The **Expectation-Maximisation (EM)** algorithm solves this by alternating between two steps:

---

### E-step (Expectation) — "Given current parameters, how responsible is each component for each point?"

Compute the **responsibility** of each component k for each data point i:

$$r_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

This is a **soft version of the K-means assignment step** — instead of picking the closest centroid, we compute a weighted probability for each component.

---

### M-step (Maximisation) — "Given the soft assignments, what parameters best explain the data?"

Update each parameter using weighted averages, where the weights are the responsibilities:

- **Effective count:** $N_k = \sum_{i} r_{ik}$ (how many points are "owned" by component k)
- **Weight:** $\hat{\pi}_k = N_k / n$
- **Mean:** $\hat{\boldsymbol{\mu}}_k = \frac{1}{N_k} \sum_{i} r_{ik} \mathbf{x}_i$
- **Covariance:** $\hat{\boldsymbol{\Sigma}}_k = \frac{1}{N_k} \sum_{i} r_{ik} (\mathbf{x}_i - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}_i - \hat{\boldsymbol{\mu}}_k)^\top$

This is a **soft version of the K-means update step** — instead of computing the mean of points in a cluster, we compute a responsibility-weighted mean.

---

### Convergence

After each E–M cycle, compute the **log-likelihood** of the data:
$$\mathcal{L} = \sum_{i=1}^{n} \log \left( \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k) \right)$$

EM is **guaranteed to increase** this log-likelihood at every step (or keep it the same). Stop when the increase is smaller than a tolerance threshold.


In [ ]:
np.random.seed(42)

from scipy.stats import multivariate_normal

class GMMScratch:
    """
    Gaussian Mixture Model fitted with the EM algorithm, implemented from scratch.
    """

    def __init__(self, k=3, max_iter=100, tol=1e-6, random_state=42):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def fit(self, X):
        n, d = X.shape
        rng = np.random.default_rng(self.random_state)

        # ── Initialisation ────────────────────────────────────────────────────
        # Random means: pick k data points as starting means
        idx = rng.choice(n, self.k, replace=False)
        self.means = X[idx].copy().astype(float)
        # Identity covariances: neutral start (spherical blobs)
        self.covs = [np.eye(d) for _ in range(self.k)]
        # Uniform mixing weights
        self.weights = np.ones(self.k) / self.k

        self.log_likelihoods = []   # track convergence

        for iteration in range(self.max_iter):
            # E-step: compute responsibilities
            R = self._e_step(X)

            # M-step: update all parameters
            self._m_step(X, R)

            # Log-likelihood after this iteration
            ll = self._log_likelihood(X)
            self.log_likelihoods.append(ll)

            # Stop if improvement is smaller than tolerance
            if iteration > 0 and abs(self.log_likelihoods[-1] - self.log_likelihoods[-2]) < self.tol:
                print(f"Converged at iteration {iteration+1}")
                break

        # Hard labels: assign each point to the component with highest responsibility
        self.labels_ = np.argmax(self._e_step(X), axis=1)
        self.n_iter_ = len(self.log_likelihoods)
        return self

    def _e_step(self, X):
        """Compute responsibility matrix R of shape (n, K)."""
        n = len(X)
        R = np.zeros((n, self.k))
        for k in range(self.k):
            # Probability density of each point under component k, scaled by weight
            R[:, k] = self.weights[k] * multivariate_normal.pdf(
                X, mean=self.means[k], cov=self.covs[k], allow_singular=True
            )
        # Normalise rows: each row sums to 1 (Bayes' theorem denominator)
        row_sums = R.sum(axis=1, keepdims=True)
        R /= row_sums + 1e-300   # guard against zero rows
        return R

    def _m_step(self, X, R):
        """Update mixture weights, means, and covariances."""
        n = len(X)
        Nk = R.sum(axis=0)          # effective number of points per component
        self.weights = Nk / n       # new mixing weights
        self.means = (R.T @ X) / Nk[:, None]   # new means: responsibility-weighted average

        for k in range(self.k):
            diff = X - self.means[k]           # deviations from mean k
            # Responsibility-weighted outer product
            self.covs[k] = (R[:, k:k+1] * diff).T @ diff / Nk[k]
            # Small regularisation to keep covariance positive-definite
            self.covs[k] += 1e-6 * np.eye(X.shape[1])

    def _log_likelihood(self, X):
        """Compute total log-likelihood of the data under current parameters."""
        # Mixture density at each point: sum over k of pi_k * N(x; mu_k, Sigma_k)
        density = np.zeros(len(X))
        for k in range(self.k):
            density += self.weights[k] * multivariate_normal.pdf(
                X, mean=self.means[k], cov=self.covs[k], allow_singular=True
            )
        return np.sum(np.log(density + 1e-300))

    def predict_proba(self, X):
        """Return responsibility matrix (soft assignments)."""
        return self._e_step(X)

    def predict(self, X):
        """Return hard cluster labels (argmax of responsibilities)."""
        return np.argmax(self.predict_proba(X), axis=1)


print("GMMScratch class defined.")


In [ ]:
np.random.seed(42)

# ── Fit GMM from scratch ──────────────────────────────────────────────────────
gmm = GMMScratch(k=3, max_iter=200, tol=1e-6, random_state=42)
gmm.fit(X_gmm)

# ── Plot log-likelihood convergence ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(gmm.log_likelihoods, color='steelblue', linewidth=2, marker='o', markersize=3)
ax.set_title('EM Convergence: Log-Likelihood vs. Iteration', fontsize=13)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-Likelihood')

# Annotate final value
ax.annotate(
    f'Final LL = {gmm.log_likelihoods[-1]:.1f}',
    xy=(len(gmm.log_likelihoods)-1, gmm.log_likelihoods[-1]),
    xytext=(-60, -30), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='tomato'),
    fontsize=10, color='tomato'
)

plt.tight_layout()
plt.show()

print(f"Converged in {gmm.n_iter_} iterations")
print(f"\nLearned means:\n{np.round(gmm.means, 3)}")
print(f"\nTrue means:\n{TRUE_MEANS}")
print(f"\nLearned weights: {np.round(gmm.weights, 3)}")
print(f"True weights:    {TRUE_WEIGHTS}")


In [ ]:
np.random.seed(42)

# ── Visualise soft assignments: colour = blended by responsibilities ───────────
# Each point's colour is a blend: R * [blue, orange, green]
# We use responsibility values directly as RGB channels (approximate)

R_fitted = gmm.predict_proba(X_gmm)   # shape: (n, 3)

# Blend colours: component 0 = blue, 1 = orange, 2 = green
component_rgb = np.array([
    [0.298, 0.447, 0.690],   # steelblue
    [0.867, 0.518, 0.322],   # orange
    [0.333, 0.659, 0.408],   # green
])

# Blend: point colour = sum of (responsibility_k * rgb_k) over k
point_colors = R_fitted @ component_rgb   # shape: (n, 3)
point_colors = np.clip(point_colors, 0, 1)  # ensure valid RGB range

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: soft assignment colours (blended)
axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1], c=point_colors, s=20, alpha=0.8)
axes[0].set_title('Soft Assignments\n(Colour = Blended Component Responsibilities)', fontsize=11)
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')

# Add legend patches
legend_patches = [mpatches.Patch(color=component_rgb[k], label=f'Component {k}') for k in range(3)]
axes[0].legend(handles=legend_patches, fontsize=9)

# Right: hard assignment colours (argmax)
hard_labels = gmm.labels_
for k in range(3):
    mask = hard_labels == k
    axes[1].scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                    c=pal[k], s=20, alpha=0.6, label=f'Component {k}')

axes[1].set_title('Hard Assignments\n(Argmax of Responsibilities)', fontsize=11)
axes[1].set_xlabel('Feature 1'); axes[1].set_ylabel('Feature 2')
axes[1].legend(fontsize=9)

fig.suptitle('GMM: Soft vs. Hard Assignments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Fraction of points with maximum responsibility < 0.9 ("uncertain" points)
uncertain = (R_fitted.max(axis=1) < 0.9).mean()
print(f"Fraction of 'uncertain' points (max responsibility < 0.9): {uncertain:.2%}")


In [ ]:
np.random.seed(42)

# ── Plot fitted Gaussian contours (1σ and 2σ ellipses) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (means, covs, title) in zip(axes, [
    (TRUE_MEANS, TRUE_COVS, 'True Parameters'),
    (gmm.means,  gmm.covs,  'Fitted Parameters (EM)'),
]):
    # Scatter plot coloured by hard label
    for k in range(3):
        mask = hard_labels == k
        ax.scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                   c=pal[k], s=12, alpha=0.4)

    # Draw 1σ and 2σ ellipses for each component
    for k in range(3):
        draw_cov_ellipse(ax, means[k], covs[k], n_std=1,
                         color=pal[k], linewidth=2.5)
        draw_cov_ellipse(ax, means[k], covs[k], n_std=2,
                         color=pal[k], linewidth=1.5, linestyle='--')
        ax.scatter(*means[k], marker='+', s=150, c=pal[k], zorder=5, linewidths=2)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

fig.suptitle('Gaussian Ellipses: True vs. Fitted (1σ solid, 2σ dashed)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Report parameter recovery
print("Mean recovery (Frobenius distance between true and fitted means):")
# Match components by nearest mean
from scipy.optimize import linear_sum_assignment
dist_matrix = np.linalg.norm(
    TRUE_MEANS[:, None, :] - gmm.means[None, :, :], axis=2
)
row_ind, col_ind = linear_sum_assignment(dist_matrix)
for r, c in zip(row_ind, col_ind):
    d = np.linalg.norm(TRUE_MEANS[r] - gmm.means[c])
    print(f"  True comp {r} ↔ Fitted comp {c}: mean distance = {d:.4f}")


In [ ]:
np.random.seed(42)

# ── Verify against sklearn GaussianMixture ────────────────────────────────────
sklearn_gmm = GaussianMixture(n_components=3, max_iter=200, tol=1e-6,
                               random_state=42, covariance_type='full')
sklearn_gmm.fit(X_gmm)

scratch_ll = gmm.log_likelihoods[-1]
sklearn_ll = sklearn_gmm.score(X_gmm) * len(X_gmm)   # score() returns per-sample average

print(f"Log-likelihood comparison:")
print(f"  Scratch EM : {scratch_ll:.4f}")
print(f"  sklearn GMM: {sklearn_ll:.4f}")
print(f"  Difference : {abs(scratch_ll - sklearn_ll):.4f}  (small = good)")

# Compare means (after matching components)
print(f"\nFitted means comparison (after matching):")
sklearn_means_matched = sklearn_gmm.means_[col_ind]   # reuse assignment from above
for i in range(3):
    diff = np.linalg.norm(gmm.means[i] - sklearn_means_matched[i])
    print(f"  Component {i}: distance between scratch and sklearn mean = {diff:.6f}")

# Compare hard labels agreement
sklearn_hard = sklearn_gmm.predict(X_gmm)
# Map sklearn labels to scratch labels via voting
from scipy.stats import mode
label_map = {}
for sk_lbl in np.unique(sklearn_hard):
    scratch_for_these = gmm.labels_[sklearn_hard == sk_lbl]
    label_map[sk_lbl] = mode(scratch_for_these, keepdims=True).mode[0]

sklearn_remapped = np.array([label_map[l] for l in sklearn_hard])
agreement = (sklearn_remapped == gmm.labels_).mean()
print(f"\nHard label agreement between scratch and sklearn: {agreement:.3f}")


## GMM vs. K-means: Formal Equivalence

K-means is actually a **special case** of GMM. Here is the informal proof:

Start with a GMM and apply two restrictions:

1. **Spherical covariances:** $\boldsymbol{\Sigma}_k = \sigma^2 \mathbf{I}$ for all k (same spherical shape for every component)
2. **Hard assignments:** let $\sigma^2 \to 0$ — responsibilities become exactly 0 or 1 (no uncertainty)

Under these two restrictions:
- The E-step responsibility $r_{ik}$ becomes 1 for the closest mean and 0 for all others → this is exactly the **K-means assignment step**
- The M-step mean update $\hat{\boldsymbol{\mu}}_k = \frac{\sum_i r_{ik} \mathbf{x}_i}{N_k}$ becomes the simple average of assigned points → this is exactly the **K-means centroid update step**

Therefore:

| Property | K-means (restricted GMM) | Full GMM |
|----------|--------------------------|----------|
| Covariances | Spherical, equal | Any shape, independent |
| Assignments | Hard (0 or 1) | Soft (probabilities) |
| Objective | Minimise inertia | Maximise log-likelihood |
| Uncertainty | None | Modelled explicitly |

**Implication:** When clusters are well-separated spherical blobs, K-means and GMM give similar results. When clusters are elongated, rotated, or overlapping, GMM is strictly superior.


In [ ]:
np.random.seed(42)

# ── Generate data with two elongated clusters (very different covariances) ────
# Cluster 0: horizontal elongated
cov_h = np.array([[4.0, 0.0], [0.0, 0.2]])
cluster_h = np.random.multivariate_normal([0, 0], cov_h, 200)

# Cluster 1: diagonal elongated (rotated 45 degrees)
cov_d = np.array([[1.5, 1.3], [1.3, 1.5]])
cluster_d = np.random.multivariate_normal([4, 4], cov_d, 200)

X_elliptical = np.vstack([cluster_h, cluster_d])
true_ell_labels = np.array([0]*200 + [1]*200)

# Fit KMeans and GMM
km_ell = KMeans(n_clusters=2, random_state=42, n_init=10)
km_ell.fit(X_elliptical)

gmm_ell = GMMScratch(k=2, max_iter=200, random_state=42)
gmm_ell.fit(X_elliptical)

# ── Compare: True / KMeans / GMM ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (labels, title) in zip(axes, [
    (true_ell_labels,    'True Labels'),
    (km_ell.labels_,     'K-Means (fails — assumes spherical)'),
    (gmm_ell.labels_,    'GMM (wins — elliptical covariances)'),
]):
    for k in range(2):
        mask = np.array(labels) == k
        ax.scatter(X_elliptical[mask, 0], X_elliptical[mask, 1],
                   c=pal[k], s=20, alpha=0.6, label=f'Cluster {k}')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('GMM vs. K-Means: Elongated Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Accuracy (compare to true labels, with best label matching)
def cluster_accuracy(true, pred):
    """Accuracy after best label permutation."""
    from itertools import permutations
    best = 0
    for perm in permutations(np.unique(true)):
        mapping = {old: new for new, old in enumerate(perm)}
        mapped = np.array([mapping[l] for l in pred])
        best = max(best, (mapped == true).mean())
    return best

km_acc  = cluster_accuracy(true_ell_labels, km_ell.labels_)
gmm_acc = cluster_accuracy(true_ell_labels, gmm_ell.labels_)
print(f"K-means accuracy on elongated clusters: {km_acc:.3f}")
print(f"GMM accuracy on elongated clusters    : {gmm_acc:.3f}")


## Model Selection: BIC and AIC

Unlike K-means, GMM has a principled way to choose the number of components K: information criteria.

Both BIC and AIC trade off **goodness of fit** (log-likelihood) against **model complexity** (number of parameters):

$$\text{AIC} = 2p - 2\mathcal{L}$$

$$\text{BIC} = p \ln(n) - 2\mathcal{L}$$

where p = number of free parameters, n = number of data points, L = log-likelihood.

**Lower is better for both.** BIC penalises complexity more strongly (by a factor of ln(n) vs. 2), so it tends to select simpler models.

**When to use each:**
- **BIC:** when you believe the true model is in your candidate set (consistent selection)
- **AIC:** when you are building a model for prediction and willing to risk slight overfitting

**Rule of thumb:** look for the "elbow" where adding more components gives diminishing returns in BIC/AIC reduction.


In [ ]:
np.random.seed(42)

# ── BIC and AIC for k = 1 to 8 ───────────────────────────────────────────────
k_range = range(1, 9)
bic_scores = []
aic_scores = []
ll_scores  = []

for k in k_range:
    gm = GaussianMixture(n_components=k, covariance_type='full',
                          max_iter=300, n_init=3, random_state=42)
    gm.fit(X_gmm)
    bic_scores.append(gm.bic(X_gmm))   # sklearn computes BIC directly
    aic_scores.append(gm.aic(X_gmm))
    ll_scores.append(gm.score(X_gmm) * len(X_gmm))   # total log-likelihood

best_bic_k = list(k_range)[np.argmin(bic_scores)]
best_aic_k = list(k_range)[np.argmin(aic_scores)]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, scores, label, best_k, color in [
    (axes[0], bic_scores, 'BIC', best_bic_k, 'steelblue'),
    (axes[0], aic_scores, 'AIC', best_aic_k, 'tomato'),
]:
    ax.plot(list(k_range), scores, marker='o', linewidth=2, label=label, color=color)

axes[0].axvline(best_bic_k, color='steelblue', linestyle='--', alpha=0.5,
                label=f'Best BIC k={best_bic_k}')
axes[0].axvline(best_aic_k, color='tomato',    linestyle='--', alpha=0.5,
                label=f'Best AIC k={best_aic_k}')
axes[0].set_title('BIC and AIC vs. Number of Components', fontsize=12)
axes[0].set_xlabel('k (number of Gaussians)')
axes[0].set_ylabel('Score (lower is better)')
axes[0].legend()

axes[1].plot(list(k_range), ll_scores, marker='o', color='#55A868', linewidth=2)
axes[1].set_title('Log-Likelihood vs. k\n(always increases — no penalty)', fontsize=12)
axes[1].set_xlabel('k (number of Gaussians)')
axes[1].set_ylabel('Log-Likelihood')

plt.tight_layout()
plt.show()

print(f"True k = 3  |  Best BIC k = {best_bic_k}  |  Best AIC k = {best_aic_k}")


## Covariance Types in sklearn

sklearn's `GaussianMixture` supports four covariance structures, ranging from most flexible to most constrained:

| Type | Description | Parameters per component | When to use |
|------|-------------|--------------------------|-------------|
| **full** | Each component has its own unconstrained covariance matrix | d(d+1)/2 | Default; clusters can be ellipses of any shape/rotation |
| **tied** | All components share one covariance matrix | d(d+1)/2 total | Clusters have similar shape but different locations |
| **diag** | Each component has a diagonal covariance (no rotation) | d | Axis-aligned ellipses; faster and more stable |
| **spherical** | Each component has a scalar covariance σ²I | 1 | Round blobs; equivalent to soft K-means |

**Rule of thumb:**
- Start with `full` for exploratory analysis
- Use `diag` or `spherical` for high-dimensional data (fewer parameters to estimate)
- Use `tied` when domain knowledge suggests similar cluster shapes

**Warning:** `full` covariance can overfit with small datasets — if n / K < 10 × d, consider `diag` or `spherical`.


In [ ]:
np.random.seed(42)

# ── Compare covariance types visually ─────────────────────────────────────────
cov_types = ['full', 'tied', 'diag', 'spherical']
fitted_models = {}

for ct in cov_types:
    gm = GaussianMixture(n_components=3, covariance_type=ct,
                          max_iter=200, n_init=5, random_state=42)
    gm.fit(X_gmm)
    fitted_models[ct] = gm

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, ct in zip(axes, cov_types):
    gm = fitted_models[ct]
    pred_labels = gm.predict(X_gmm)

    for k in range(3):
        mask = pred_labels == k
        ax.scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                   c=pal[k], s=10, alpha=0.5)

    # Extract covariances and draw ellipses
    for k in range(3):
        mean_k = gm.means_[k]
        if ct == 'full':
            cov_k = gm.covariances_[k]
        elif ct == 'tied':
            cov_k = gm.covariances_          # single shared matrix
        elif ct == 'diag':
            cov_k = np.diag(gm.covariances_[k])   # diagonal matrix
        else:  # spherical
            cov_k = gm.covariances_[k] * np.eye(2)

        draw_cov_ellipse(ax, mean_k, cov_k, n_std=2, color=pal[k], linewidth=2)

    bic_ct = gm.bic(X_gmm)
    ax.set_title(f'covariance_type="{ct}"\nBIC={bic_ct:.0f}', fontsize=10)
    ax.set_xlabel('Feature 1')
    if ax == axes[0]:
        ax.set_ylabel('Feature 2')

fig.suptitle('GMM: Four Covariance Types (2σ ellipses shown)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nBIC comparison (lower = better fit per degree of freedom):")
for ct in cov_types:
    print(f"  {ct:12s}: BIC = {fitted_models[ct].bic(X_gmm):.1f}")


In [ ]:
np.random.seed(42)

# ── Retail synthetic data: TotalSpend vs. Frequency ──────────────────────────
n_customers = 500

# Segment 1: Budget, high-frequency shoppers
seg1 = np.random.multivariate_normal(
    [60, 28], [[80, -10], [-10, 20]], 180
)
# Segment 2: Premium, low-frequency shoppers
seg2 = np.random.multivariate_normal(
    [280, 4], [[1800, 50], [50, 4]], 200
)
# Segment 3: Mid-tier, moderate frequency
seg3 = np.random.multivariate_normal(
    [150, 14], [[400, 100], [100, 30]], 120
)

X_retail = np.vstack([seg1, seg2, seg3])
retail_perm = np.random.permutation(len(X_retail))
X_retail = X_retail[retail_perm]

# Clip to realistic values
X_retail[:, 0] = np.clip(X_retail[:, 0], 10, 600)   # spend £10–£600
X_retail[:, 1] = np.clip(X_retail[:, 1], 1, 50)     # 1–50 visits

# Standardise before fitting
scaler_r = StandardScaler()
X_retail_scaled = scaler_r.fit_transform(X_retail)

# ── Fit GMM from scratch ──────────────────────────────────────────────────────
gmm_retail = GMMScratch(k=3, max_iter=200, tol=1e-6, random_state=42)
gmm_retail.fit(X_retail_scaled)

R_retail = gmm_retail.predict_proba(X_retail_scaled)  # soft assignments

# ── Build a DataFrame with spend, frequency, and segment probabilities ────────
df_retail = pd.DataFrame(X_retail, columns=['TotalSpend', 'Frequency'])
for k in range(3):
    df_retail[f'P(Segment {k})'] = R_retail[:, k]
df_retail['Dominant Segment'] = gmm_retail.labels_
df_retail['Max Probability']  = R_retail.max(axis=1)

# ── Show top 5 most uncertain customers ──────────────────────────────────────
print("=== Top 10 most uncertain customers (near segment boundaries) ===")
uncertain_df = (
    df_retail
    .nsmallest(10, 'Max Probability')
    [['TotalSpend', 'Frequency', 'P(Segment 0)', 'P(Segment 1)', 'P(Segment 2)', 'Max Probability']]
    .round(3)
)
print(uncertain_df.to_string(index=False))

print("\n=== Top 5 customers most confidently in each segment ===")
for k in range(3):
    top5 = (
        df_retail[df_retail['Dominant Segment'] == k]
        .nlargest(5, f'P(Segment {k})')
        [['TotalSpend', 'Frequency', f'P(Segment {k})']]
        .round(3)
    )
    print(f"\nSegment {k} (top 5 by probability):")
    print(top5.to_string(index=False))


In [ ]:
np.random.seed(42)

# ── Visualise retail GMM results ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter coloured by dominant segment
for k in range(3):
    mask = gmm_retail.labels_ == k
    axes[0].scatter(
        X_retail[mask, 0], X_retail[mask, 1],
        c=pal[k], s=20, alpha=0.6, label=f'Segment {k} (n={mask.sum()})'
    )

# Overlay fitted ellipses in original (unscaled) space — compute via transform
# We draw ellipses in scaled space but show axes in original units
# Use sklearn GMM on scaled data to get covariances in scaled space
sk_gmm_r = GaussianMixture(n_components=3, covariance_type='full',
                             max_iter=200, n_init=5, random_state=42)
sk_gmm_r.fit(X_retail_scaled)
means_orig = scaler_r.inverse_transform(sk_gmm_r.means_)  # back to original units

for k in range(3):
    axes[0].scatter(*means_orig[k], marker='+', s=200, c=pal[k], linewidths=3, zorder=5)

axes[0].set_title('GMM Customer Segments — Retail Data', fontsize=12)
axes[0].set_xlabel('Total Spend (£)')
axes[0].set_ylabel('Purchase Frequency')
axes[0].legend(fontsize=9)

# Right: stacked bar chart of average segment probabilities per dominant segment
avg_probs = np.zeros((3, 3))
for k in range(3):
    mask = gmm_retail.labels_ == k
    avg_probs[k] = R_retail[mask].mean(axis=0)

bottom = np.zeros(3)
for j in range(3):
    axes[1].bar(range(3), avg_probs[:, j], bottom=bottom,
                color=pal[j], label=f'Seg {j} prob', alpha=0.8)
    bottom += avg_probs[:, j]

axes[1].set_xticks(range(3))
axes[1].set_xticklabels([f'Dominant\nSeg {k}' for k in range(3)])
axes[1].set_title('Average Segment Probabilities\nper Dominant Segment', fontsize=12)
axes[1].set_ylabel('Average Responsibility')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nInterpretation: a taller 'own-segment' bar means the segment is well-defined.")
print("Mixed bars = customers near boundaries with genuine uncertainty.")


## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** In the E-step, you compute responsibilities r_ik. What does it mean if r_i0 = 0.5, r_i1 = 0.5, r_i2 = 0.0 for point i? What does it mean if r_i0 = 0.99, r_i1 = 0.01, r_i2 = 0.0?

<details><summary>Show answer</summary>

When r_i0 = r_i1 = 0.5: point i is **on the boundary** between components 0 and 1. Both components explain it equally well (given their current parameters and weights). This is maximum uncertainty between those two components.

When r_i0 = 0.99: point i is **deep inside component 0**. The algorithm is 99% confident it came from component 0. This is typical for a point near the mean of component 0 that is far from the other means.

In both cases r_i2 = 0.0 means component 2 explains point i extremely poorly — the point is far from component 2's mean or in a low-probability region of component 2's Gaussian.

</details>

---

**Q2.** Why is EM guaranteed to increase (or maintain) the log-likelihood at every iteration? Can EM get stuck in local optima?

<details><summary>Show answer</summary>

EM increases the log-likelihood at every step because of a mathematical property: the E-step computes a tight **lower bound** on the log-likelihood (via Jensen's inequality), and the M-step **maximises** that lower bound. The true log-likelihood is always ≥ the lower bound, so it can only go up or stay the same.

Yes, EM **can get stuck in local optima**. The log-likelihood landscape for GMMs is non-convex — there can be many local maxima. Mitigation strategies:
1. Run with multiple random initialisations (`n_init` in sklearn) and keep the best
2. Use K-means++ to initialise the means more intelligently
3. Use `init_params='kmeans'` (sklearn default) instead of random initialisation

</details>

---

**Q3.** You have a dataset of customer transactions and you fit a GMM with K=3. The BIC keeps decreasing from K=1 to K=8 without any clear elbow. What might be happening, and what would you do?

<details><summary>Show answer</summary>

If BIC keeps decreasing, several things could be happening:

1. **The data genuinely has more than 8 clusters** — extend your search to larger K
2. **The Gaussian assumption is wrong** — the data may not be generated by Gaussians at all. Consider non-parametric methods (DBSCAN, kernel density estimation)
3. **Wrong covariance type** — try `full` if you are using `diag` or `spherical`; the model may not be expressive enough
4. **Outliers** — a few extreme points might be causing additional spurious components. Remove or cap outliers and re-fit

What to do: (a) try K up to 15–20 to check if a plateau eventually appears; (b) visualise the data and fitted Gaussians to see if the additional components look meaningful; (c) involve domain knowledge — do 8 distinct customer segments make business sense?

</details>


## Key Takeaways

1. **GMMs model data as a weighted sum of Gaussian distributions.** Each component has a mean, covariance, and mixing weight — together they describe a cluster's location, shape, and prevalence.

2. **Soft assignments (responsibilities) quantify uncertainty.** Instead of forcing each customer into one segment, GMM assigns a probability vector — more honest and more useful for targeted marketing.

3. **EM is an iterative algorithm that alternates between:**
   - E-step: compute responsibilities given current parameters
   - M-step: update parameters given current responsibilities
   - EM is guaranteed to increase log-likelihood at every step

4. **K-means is a special case of GMM** with spherical, equal covariances and hard (0/1) assignments. GMM is strictly more general.

5. **Full covariance GMMs handle elliptical clusters** — they can learn clusters that are rotated, elongated, or correlated, which K-means cannot.

6. **Use BIC to select K.** Unlike K-means (which needs the elbow heuristic), GMM has a principled information criterion that penalises model complexity.

7. **Four covariance types** (full, tied, diag, spherical) let you trade flexibility for stability depending on your data size and dimensionality.

---

**Next:** Notebook 7 — Dimensionality Reduction: PCA, t-SNE, and UMAP for visualising high-dimensional customer data.
